In [ ]:
# ── Colab Setup ──────────────────────────────────────────────────────────────
# Run this cell first. It mounts Google Drive and adds the project's src/
# directory to Python's path so grammar_loader can be imported.
# If running locally, it just makes sure src/ is on the path.

import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # ← Update this to match where MScProject lives in your Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/MScProject'

    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {os.getcwd()}")
else:
    # Running locally — add src/ to path if needed
    src_dir = os.path.join(os.getcwd(), 'src')
    if os.path.isdir(src_dir) and src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print("Running locally.")

# 03 — Layer-wise Probing

Trains **linear probes** on the frozen hidden states of a trained model to test which
layers encode structural properties of the grammar.

Barry's notes:
- Probe each layer separately
- Lower layers → local patterns; higher layers → abstract/counting structure
- Ignore PAD tokens; keep CLS + SEP
- Probing is diagnostic — no train/test split needed here

Requires a checkpoint from `02_train.ipynb`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

from grammar_loader import load_grammar, build_vocab, tokenize

# Re-use class definitions from 02_train.ipynb
# (copy here so this notebook is self-contained)
from torch.utils.data import Dataset


class GrammarDataset(Dataset):
    def __init__(self, filepath, vocab):
        self.sequences = []
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids = tokenize(line, vocab, add_special=True)
                    self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def collate_fn(batch, pad_id):
    max_len = max(seq.size(0) for seq in batch)
    padded = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(batch):
        padded[i, :seq.size(0)] = seq
    return padded


class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=256, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.cells = nn.ModuleList([
            nn.LSTMCell(embed_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x, return_hidden=False):
        B, T = x.shape
        emb = self.embedding(x)
        h = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        c = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        layer_outputs = [[] for _ in self.cells]
        for t in range(T):
            inp = emb[:, t, :]
            for i, cell in enumerate(self.cells):
                h[i], c[i] = cell(inp, (h[i], c[i]))
                inp = self.dropout(h[i])
                layer_outputs[i].append(h[i])
        layer_hiddens = [torch.stack(steps, dim=1) for steps in layer_outputs]
        logits = self.output_proj(layer_hiddens[-1])
        if return_hidden:
            return logits, layer_hiddens
        return logits


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2,
                 ff_dim=256, dropout=0.1, max_seq_len=512, causal_mask=True):
        super().__init__()
        self.causal_mask = causal_mask
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        self.num_layers = num_layers

    def forward(self, x, return_hidden=False):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.embedding(x) + self.pos_embedding(positions)
        pad_mask = (x == 0)
        causal = None
        if self.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)
        logits = self.output_proj(out)
        if return_hidden:
            return logits, out
        return logits

## Configuration

In [ ]:
CHECKPOINT  = 'checkpoints/anbn_lstm.pt'   # checkpoint from 02_train.ipynb
DATA_FILE   = 'data/anbn_n1-20.txt'
GRAMMAR_FILE = 'grammars/anbn.txt'

BATCH_SIZE   = 64
PROBE_EPOCHS = 20
FIGURES_DIR  = 'figures'

## Load checkpoint and rebuild model

In [ ]:
device = torch.device('cpu')   # probing runs on CPU for simplicity

ckpt = torch.load(CHECKPOINT, map_location=device)
vocab      = ckpt['vocab']
model_type = ckpt['model_type']
model_args = ckpt['args']
pad_id     = vocab['[PAD]']
vocab_size = len(vocab)

if model_type == 'lstm':
    model = LSTMLanguageModel(
        vocab_size=vocab_size,
        embed_dim=model_args['embed_dim'],
        hidden_dim=model_args['hidden_dim'],
        num_layers=model_args['num_layers'],
    )
else:
    model = TransformerLanguageModel(
        vocab_size=vocab_size,
        embed_dim=model_args['embed_dim'],
        num_heads=4,
        num_layers=model_args['num_layers'],
        ff_dim=model_args['hidden_dim'],
        causal_mask=ckpt['causal_mask'],
    )

model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded: {ckpt['grammar_name']} / {model_type}")

## Hidden state extraction

In [ ]:
def extract_hidden_states_lstm(model, batch):
    """Returns list of (B, T, H) tensors — one per LSTM layer."""
    model.eval()
    with torch.no_grad():
        _, layer_hiddens = model(batch, return_hidden=True)
    return layer_hiddens


def extract_hidden_states_transformer(model, batch):
    """Returns list of (B, T, H) tensors — one per Transformer layer, via forward hooks."""
    model.eval()
    layer_hiddens = []
    hooks = []

    def make_hook(layer_idx):
        def hook(module, input, output):
            layer_hiddens.append(output.detach())
        return hook

    for i, layer in enumerate(model.transformer.layers):
        hooks.append(layer.register_forward_hook(make_hook(i)))

    with torch.no_grad():
        B, T = batch.shape
        positions = torch.arange(T, device=batch.device).unsqueeze(0)
        emb = model.embedding(batch) + model.pos_embedding(positions)
        pad_mask = (batch == 0)
        causal = None
        if model.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=batch.device)
        model.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)

    for h in hooks:
        h.remove()

    return layer_hiddens

## Probe target

At each position, the label is: *how many `a` tokens have been seen so far?*
This directly tests whether the model tracks the counting property required by aⁿbⁿ.

In [ ]:
def make_probe_labels(batch, vocab):
    """(B, T) integer labels: cumulative count of 'a' tokens at each position."""
    a_id = vocab.get('a', -1)
    labels = torch.zeros_like(batch)
    for b_idx in range(batch.size(0)):
        count = 0
        for t_idx in range(batch.size(1)):
            if batch[b_idx, t_idx].item() == a_id:
                count += 1
            labels[b_idx, t_idx] = count
    return labels

## Linear probe

In [ ]:
class LinearProbe(nn.Module):
    """A single linear layer trained on frozen hidden states."""
    def __init__(self, hidden_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc(x)


def train_probe(hidden_states, labels, pad_mask, num_epochs=20):
    """
    Train a linear probe and return final accuracy.

    hidden_states : (B, T, H)
    labels        : (B, T) integer class labels
    pad_mask      : (B, T) bool — True where token is PAD (excluded)
    """
    H = hidden_states.size(-1)
    num_classes = int(labels.max().item()) + 1

    probe = LinearProbe(H, num_classes)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    mask   = ~pad_mask                  # valid positions
    h_flat = hidden_states[mask]        # (N, H)
    l_flat = labels[mask]               # (N,)

    for _ in range(num_epochs):
        logits = probe(h_flat)
        loss = criterion(logits, l_flat)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        preds = probe(h_flat).argmax(dim=-1)
        accuracy = (preds == l_flat).float().mean().item()

    return accuracy

## Collect hidden states across all batches

In [ ]:
grammar = load_grammar(GRAMMAR_FILE)
dataset = GrammarDataset(DATA_FILE, vocab)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, pad_id),
)

all_layer_hiddens = None
all_labels = []
all_pad_masks = []

for batch in dataloader:
    batch = batch.to(device)
    pad_mask = (batch == pad_id)

    if model_type == 'lstm':
        layer_hiddens = extract_hidden_states_lstm(model, batch)
    else:
        layer_hiddens = extract_hidden_states_transformer(model, batch)

    labels = make_probe_labels(batch, vocab)

    if all_layer_hiddens is None:
        all_layer_hiddens = [[] for _ in layer_hiddens]

    for i, h in enumerate(layer_hiddens):
        all_layer_hiddens[i].append(h)
    all_labels.append(labels)
    all_pad_masks.append(pad_mask)

all_layer_hiddens = [torch.cat(hs, dim=0) for hs in all_layer_hiddens]
all_labels    = torch.cat(all_labels, dim=0)
all_pad_masks = torch.cat(all_pad_masks, dim=0)

print(f"Collected hidden states for {len(all_layer_hiddens)} layers")

## Train probes and report accuracy

In [ ]:
print(f"\nLayer-wise probe accuracy ({ckpt['grammar_name']} / {model_type})")
print("-" * 40)

accuracies = []
for layer_idx, hidden in enumerate(all_layer_hiddens):
    acc = train_probe(hidden, all_labels, all_pad_masks, num_epochs=PROBE_EPOCHS)
    print(f"  Layer {layer_idx + 1}: {acc:.4f}")
    accuracies.append(acc)

## Plot results

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(1, len(accuracies) + 1), accuracies)
ax.set_xlabel('Layer')
ax.set_ylabel('Probe accuracy')
ax.set_title(f"{ckpt['grammar_name']} — {model_type}")
ax.set_ylim(0, 1)
ax.set_xticks(range(1, len(accuracies) + 1))

Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
out_path = f"{FIGURES_DIR}/{ckpt['grammar_name']}_{model_type}_probe.png"
fig.tight_layout()
fig.savefig(out_path, dpi=150)
plt.show()
print(f"Figure saved to {out_path}")